In [0]:
import base64
import gzip
import hashlib
import io
import tarfile

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("silver_update_id", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
assert RUN.startswith("dq4_silver_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SILVER_UPDATE_ID
LANE = "dq4_scans"
PAYLOAD = """H4sIAAAAAAAAA+xcXXMbx5XVs37FvKRIbohxf394HVfJNtdRSpFSkrypra0thKIgmQkJ0AApx5Xkv+85t3vAmYFkDE3WZrdq8SBBAKb7Tve9555zb482P1x8dnW62Zj52cX58vzs9OL2zeZstV60mx8uHt3vpfAKwfFvHb3q/8232nrzSHvjrIk2GvMI3wZrHjXqnvNOet1srk/XTfNocXq9Wr759O/2ff9/9PX1y5Mnr0+aFy+blyd/ePbk65Pm9ZOvnp00f0rzt4sPf2o35xcfFuv5D2fz68ur9u0Pbv7hfHXxKWeZ8wf1EqNMUMn45smrx398+vq3zbvm8Gy1PFtcXc//svjpuLlZnl8fN+v5xYp/fn9+3FzL+2t5v5H3G3l/Je+v5P37xfLtYn30uMG4zeG/P3n23Qn+Plhdrq4+t0pFnd3BcXMwny9Xy8V8jvdfP3n1+vD5d8+e8YpvXnyH2zv61Kfat2r7p26D1s7rYEw28E18FNscg9IhRGeVC8YdN/b2isGYr16/fPr826OjY5ja2ReNStbbexg4k+lkNrx1XsWQUzQBARX9cRNaZZOJybgYfUgmHDdK7FPT7Yv3WUCZR/6Y6dYFnZNPJitlvNEGdrQREZ98zl7hm6Drj+0dzPMPZZ5XNphoPMaN1uZM81JyJmDtLEwMKsdfYF98KPuSsjloa5WH95kM/zOtDT5ln5RJ2mZr6o/NHcxLD2VecMm6oLziZvrkxTxjtbLOGwUgj8r9guXL94kObCrN8MX9TEgO66e5aHBBWSf4HT4JTqusENp3NzCph1lABV+D62nsbPIm+Si+lpEHvVYhhuRDCP4X2KcfaoONVTEGl5PD9mYn8REMli96nXNSSbs7rZ9TjtH1YPZhd2PwUbuYgXlZdl5jQQlh1ihY15nnJpnnktZApQdCZ9diGwF1wcWENTSa6JxDdtG4jOQYvUsdOu9bvs367POb9fLzs8V6uVh//vbsan6+vLq5nq8X7+bnb/+uAVU5OuzKfYxnkivJDE4k9xBM8G1ksIRssKSRtiblWw/wQehrY3wX4xqrp+95H9Yp7SPc/l4YoIFHggNYdCtYkDMSNBDTIPkgLWb4hXHwZLh4sgqfArC2m5Gj3ZfN990HEi+23d1vO+QebMUyHTT8BYZyN4KlS/vgQ0auQuYHEwmMzgmuPjJ98WGxvJ6fwYV0jEgp5j7pYWRyQgJQAAzjSPGT2AwfxabgZpDPrL2nzdYY4Dnu/uEICwgkjIUHeSeEr8WKwDmAzlkl8JU8EfB+3mRs5IMlEQ8CY+AEgBZlbZIltVhm6x1Yl8Ie2Hva7AI8I2R9L94asXp0DDiyyuIhCfS51VZZRBxQ0pMeJqw/c2Dwmi4S7TYotb0juGzNR5xoE929eNmQdmetALVwawC5tSER2ZGLIuQOtjcALHWXuO8KiVurU0aiTSk/GJu0KsMpQGl90h5Jko6CvKRBjVLEPCBs+x0FRr853SxEe8H8VyfPTr5+3ZytTi8Wm7OF6LvT65pPDyjWrg/fnV9cL9aH79ary/mfN6vl4fWq/F0U49nq7QKmHpyu16c/fbG5Xt+cXX+BD8+X7+W7Bh/h/XFTP9v8tLleXG4/Pd/MN6ub9dmiebNaXSxOl19+eYDh/trMvmz+2g6v+U1zwJUW4zolev6WP1dHbW/KI05WboS7c3B8e4Nlrvlg3P7+8E7+/vELFoPfHR1xDft6GMtZX9uLlzcXF+fvDi9WP2IBccOXh9TM9YbnH04vbhZHnFGmvR2bI4u4vh1SNlMumC9vLt8s1j0/4dsPkNeLNXYFC4Kr/u3li983bn61Xr2tFYD2E4r/8dHj6gRv2oG6f9N2JpytbpbXh2/aDzLTkgYs1udn/Gpzc3kI006aP/725HnzrmUxoHn6qnn+4nVTXO/5Nw0vbb7ovn3xsuG/v5R/f39+dHuPTfOao+jm5BlGVM3J82/qjOvT5fvF7ny9t90U16MpWIv4yLC9SX/dDO5g87N3sBkNvxnfwSdv5FMzdiNfjUa++rjhzZe/acyUJcNebhbLzc2Gyybeo/jFV0+/ffr8df3Nj+sVgqAUZH7mZ+UHi7ePxasEPt48/t2Lp1iu5gXXrOc3iNGBH8nyvRNXkq/45vFj3PnLE7nV3ko//vbli+/+0Hz1Hx/3xH/9Z1fa/ne+Nrv136vT6+9XF6v3P4HNbm4uru9dAf75+m9gZWhY/6U60v9f//2feN23/jt2ln9yBVgrBRoj5PrycnXx2cVBVwGQEpluWaYiJ4M2671AkHyrtbNKGx8iRKF1JJ1tzM6BS1PHe4gsVhhcN9i+egJ4Lsig8Det/iX0bIFaF1KGYQZmUJfa1u18NsNQvrXW2xBYTwWV5l142wawezB8DxHg02j4vfYZF1UU7fSf5zf/JfbdTWgLoQ9R2LyNYI0xQEYGiL7kKVG9b3OE1gkKatsl0xXLY6kz3MXAal/h1CEXJq5lmGALLfepDUGxhKC8107DABta/Mto76NRMXPZfH+EvRYEaG1taMH7Or/GjeU2F4mb02Cz+I3So4+wy3ArmyDbvfUhRO4oPCtBMWDnoDejKzpnO/IUs7Cgu57lDTQTPXSGFd5xI/iLH3zGH4L5Y/2SsSlCcHkW6XMwAVezuYCZrB4PPsW8pO6lFpPIEFOLUFpD2iLyIFYgu3An+IihmlrZWGW2deY4zbHYLkli4Gc3t24FF8X+isLzO4sXipMhdbXeGYtbZFme5Q6NSX32cFVDo8aj7TPGwGFUclJ3+cvVqVgTWGgTqLGtpUfwTqW+CSkOxRa9RcKMbJ9keBfAiXUWvDSryKalkivX760ZKwXDdwBTc+ZYih3DpUixK7AzFmJMlrGG/YlGDEwATw896Sgp8anETDfafmsQuyHdxtudendSeHLSAkitY2xlnxJr6fR0F2Gvd3BurBvuOvFXhI8wqbaOnbU2j1wmA+ewJ4LQLoWW9XpsgXMFIJxH4mgT5kPYe6ARwQqA0kZQHk9A8HY80hRDfGnibDfMVFgs6FrCRrtU3NAzboAwOhkHxPFS321tQlYLGZtlnCQT0xtigg0ulKrbrw6kjsAbkNABcmjVe2ka5EtF0LcpGWQCxA84X5SCj3FYtEC7MtKCYZUwtrEbbr8hgHU3XAztyn3L7WcnNSctfyWgrgcjiQa3zlINv8mtBDMpZ1aRSTT2B5hggc3ZVijOt1CcBIYVG06Dl+CDHX1oCXOtM8khkUfsSrDOSDs3xQSIy8AIDcwP/cEnmMZtH7msYzZsE7HMIUj6L7qn3PQswlVaMp8YmDjZZ2RPFHkC4AM4dNvO43a4O1iDhdLmdqVYO5PWzXBVDL8BDxtmU3ZF8caCW7CF4r38LFj6N8ZH8rdMWAix1I283zL6s9/yoMvbjKB8tCzW1hqaTiWmclKVvzmXnMoABu6atYtfs6OIj6Whl23MMaSsXPn8I2NOsc0VDtmHHcCsKQXrQbAx2owkj5lF/ABkcqCDOuk+OcGdnJAweWjAbE2qw02xBQted9C2A05mVFeQ3nYA4F8xshOXA2JUJ9ZEHZNGTtZFByTUuUsPZvJy6FRQ+OL94d/OVlfni80/jnpbllMuZzEQZMO1sZKEBIgMXJgnIoxnryiTPoBPGGRPNrayV1nTcTTxeDvgHWzrDOvZFQI7NYX2w4sISq4QZ/C/xATawlPAsCxbDPQ0g9AHUKSggrW7o0wwxzkjGL05qOwSRCUTPIxML3+BxMMXNIR3axLkGkFAg2Qopk+2zCJMEWKquusnzByQ/buZS2IpESMLacoqOCgd0GePrXcK9BiTYGGAe6Sf0cMIeodWvcsnzIzQdLtgrHVlLHaIu+WAgu9C2SNQAkLWKCtcJiNDR7gFhoWSM4PB9tvikdPMOHJ9ljTp/Iie4+ahQMeiNANekTHB8SxZrqHCQGJQmrTCayiaPBx4glUgv/fpwM8klZoS4s5bEAEHG9votEgFfASQUClR+yHSdMmCDKc0kVzAJcKYnEOWa7ow2/7D5CmYJ3GdsDPIVhr0CPkzaa4pckNK0prOPngzGm2vLXAFkP30qaAGa7AFHWaFRurirzNsB5ZDtezCBafJ/DK7QkkzN7BBne3uIPsPeGivfSFePbaeBFfZ0EFo72AeUH/gVlFiP0VY4IHBWJVIb2fPG6uC3ORgpXLlNJnuhp5gGvArpb5Yl9aX3WZPYYFJ3GeGiSDWTdAOhBybBTcGQW49cFCbSA2QTYWLboAJFoBAluMby6Ie6tE89hTFAlPwx9Y+XrQBsgoEUtH2zB0BR0cyTAi3YDMkRBwPMsUK0Ox7HmCQECsr5ZMCX0Z6YFcRewMjPTUNOS3bpaCztoQY/d1MP4fotepvVq2LdMymmGCK0gMkt+waRngK1J7AJt60XCSqPwtnJ4WL/TGmGeHzWHpKlchIezJqgWE52gODIhaAZ6Y0GKDKSRCa580gPjP2UUctHXsbuhEmmRCM3i0XWhlFtyyCDOMJ/ErSJ7hNAvIp56jFfdHdiTUB+DTVRakOwOzUDTfNHLtb8DLQbiVnSuecwCLoC5cEiwNxgQxKkLUEE+RxxZMiAIrsWCBw/RGmmVBKIzeDFakxVEZLNZ5YR2UhIvOcWOKNxtbirUNsQeGFnCoKTA0ezJ6KSNjOXq4PxbeqNCvFmGxbEeUZG+OhmeiCWCFWRCkbTEyWp3oKttYRJpkA7xq5hFQzRegjIgaSJBUuFXYKysgESDsuElPl+AQ9hCenJKIT3JW5vcNvP0n7i3FajFt2xhVMc3V7dCkB1CNoyMHYBwNayrQsh0McD0LhI2g4UKVA7+yPsN8EcCGnUqf9C/mQZApemuV9JnIgdEzgaVr4E+tlxSoI7AQlSUiLpfYRu8unzMwzZ7Iz73uhiqRRKsAQyeT01Qu05BmgY4ucguiA1SKgo00wFKgF8m8RrXJSg6R7O9AUS7wv5xiRaXriA68Cn+KnrmCpBx5ATyCLOYV7J3YixRom8ha+QRU7un6KATGVuvRAiskWgh/JhhYEr6UXJgvDc3zgmJFgxXtFnoPQB3xDjkk1pnf5FBuoW8bbwUPBBSMEcJyOFcKAmM47WqFA+zWdHtjdash42BCCgJUvBYhukP0nShW55sAGK4cUbQ0BngQ0bhCZmrVxPfxsBhBLPOkoeitIr8IHRe5EZIMk0UG0KkFuO/oE8yJYzsA8V0rDTkow8JOhIsG3WK8s7CRKHcGSnhnZHix4QF7LINUQimBuoaLHdsD9BiFhbf1mNnIccORU/deKNsUuFQ4Csd5GRBGIEh+HYD4OQDaETVBQ9zoFlsirVutGmWAMJacQfejN9m8/vjn7x7aTodvu7H1lJBDIWeIHdADR54j2yH8ZRDZiSTzCzMfB1XcwYCQXTWsGk0c2AsF4NBY/wGOJdoDWGJIchAayZD+4dsLUNgc3SHS1bFs8f6b7unlmiUw68DIDBCVcMU55ei7xdCJzyjY9uqnBw+yUPg4i8HtbUVX+WfYgMZp5qh7aU4P0McxzRCwknwH2EF6FuPneABPMCKbUTd71BLt0mjAbdYjmieM+EeNZajg81AvUMWi9z1KvgQNm+TBY4y2lp49MLXW0CZaQ4NKSq0HqL4io22G3jy4fpaEM/0D+twG3H1n9KPjK89/4kUeoMtfUdZmGrjTFlbN/d2+EFHHuC1fMPNQKygJIi45qD0sHQ4K14K+BxS1jtq7jpzpOZ9zl3a2bWWkbx1KS0WBqUjptoSR0KAdYLQhCBDlpJeH7EtZYbeemSUEHPqxVHNCk2sKsRKHd5fXgSuPPZtxETVEKgAGhVLTEktIn4LJDToClztUapp/EsGEb5r7PwXBvFLgVDfLlnlij43bGUoiFrLasWiJSAXBsERpDzE4swGC5hbFSLIODIXp5Ee4nsCrt0sS8BoFcals9jiztHFv5oHRdSisutZgfRDOwmF8OnmIDImt1Xp7bkOAoVc06wIQnSQzy4m4dBDcoklEV5RiKckzQyWCjuF0EZCzdBnBixbzGTILVkt+F2I0w5VEWVQ93vO9XwWMBT7mXXPkUGAiEOYQadoPlF2xLi8zhKdAA3Y4xaXPv+imzWxPGXVsjC8AioSEPHvGKrXBNyrLL6skEki261YOVg6tAUcZgi4wui2EmFHzluR5fHhu8ef/Z+y6HQJfW3F16kS7U1Aa5BhHiHRyCnisPBkq3XBkAObw3qtLA9reD7Dciej5n2Cv/FF2zrT/lcv5cx9KODAqqjUIRcxqsJjGbKYudABJ2gJIzLKn0R5lgRQgVekZNEqmNFuGq2mFbqzbV3fDDKPAYWovcQrXiU4VqoDTP2SROBWKawnD4aTaW6s+ujbY2KfKobJgllxRmQiqGeCIMBlNaylBcmQ+uaYoM4JEZDrffJjYfzC5iuxIYs8IldCkxzSJWILPbR281nmU6nYA2mVqGZ8ACkTr0BthvAMzPt03A2/ql1bXaIMFs6/zydJfJFmIhO1YZgJ1t5JJBG/JTXefvrr+LAXV+U+StK/dc0DTU+imSo+IzlmRZQDISUpBeOgjyFKS+sp2w7waYYEBC5r1FNJnel7qgyIFQYB0zMxWSDLKywIfCguHToEBRljRYEDquz7T5ifkk59or2e5+uV4XOo2cO8SzKOl2GDBBBAEkAh+QhSlZutTYscDTBYgg6wEsSSqreTv0XttI+lMYMPU79ktUd0JjFtnRVixkRqAwczOwEFgdLRkVEkbQqdv3MI2LQW4Bykd5qJQRnIyQi5DwoDw8jwNyEHmgqmRhlu4gL/mAsdXlAKHrLt8/teV5xN1iVH0O0G//YDkZ8QHZqrE77IhEaTgkVlwM9TZwzg2vnjA7+xSDG3dSc+FjlTybFwbwVR6ON+MqHTRs4DkhkgE4sNQLoSKYq5GMoTScLQ/m9gafYFqIZqewLdVR1dUjbZXcEDaJ7Wb4LBgZoxbqzULiK8OHiRBQrh5UstPCWDtkOD0SMeINtURoRn1GxA2PLpVWNI9XRMhrJOdURAsAHOvMI3P4I5ajdmY73ARzEuTPrsa2rj4mOGTdSlhkqwTkwD7A45EaVEzWyQPIACL6LNJedghmOxhtgjHZpEHbQ5IUhDXrBbMwasom0VI8xxORRZAzA5is1jbxdHDkk2rAfriv4cO3aTjYflu8ilkNMU8iJhYtNjPl3FTWtYaneMCVj10nnl4uOQ9mkdnzOTH+TxXdCdtujAlGMIzkPMVAJZVzGWFnOdS40J3LWRNIcJiLKAp8bk+EpM0QnCBRmiUwGS52Q08wC6TMjlKRVqWqKlkoi8bFtiNnAUidDcAVVpQzjElQjdGRCnSnDeq1Eya2Jo77QXJ8sRyvHvG2WJpEOx4sHW94J5/gB9gGPkEvDI//FQb4JWtVSmhb+R9E/CRUoW1j0VW2qyBeGJghtKx+xRp4VqyVBHAMzZoKmFRIKUR8AJ4LHtxV48twE6xxCMkBX6qnEH3hK6X6bWoR3Gp5+DeCn9hM0kC4RzoEWYqsgQv1Lwn8doy7GDGwwdczJkYovyn0ZQaqSsoAocP/DSKICQRgQBz/B4HMjpmUzHtDTDHB5R0lJtLJlTbheFNMOcI2AGCeanAuRtbWpKZajp84mMkyA8i2Ko8jO9cNPcWy2kz8VefG7JiVdJwHVknxNtVFQvaDtmY0R+XkHIDm0VrIdc2MCIqS3H+3d63NbRtZNp/zK/hli/KOhKDRABpwMlOVsZ2sazdOYjvzIVO7XIqCFFZEiUNSjr2VH7/n3NsAAZASQUmRHRudh2wIaFz04z7Pve092dpfB0pg7toNaAC/q1QPEvFcKAVMFAarJUYUViF5fEJHuc0IqcDGEQM+rvewmwLxd7W0pNTJaEChtfxKiyGKFRIVMKqaEjxHrBhdFpazxtg+zC3nCO+TTpKylw4UQC1oqiuKVnbCLrK0hFVD6kU2Iz7DYdzEkMmgu2VgphA2OTGZzad3vxrLKnabIlh8m4Rxb4OoRgqYih3R3ODuaUxDQ6NlUJVAB77IEUGSm3p/HahJoEgMPTrM1LzcdQA3y9/kMf02UEzBRmNSleRErBoLldzlkC2xKATrDjq8O/Vxu6vaOoBktaWPQMRnrG4YMK00CSgEQ1YyiukroXeZwQD62GD+pUklnHwfHWggpndLgF2lvsh8H+AHc5JwFCy6kDhi/obIdphfVH4I3VcmUz2+++3EFap2eFapYrkP2TZZlRUdXZyFsBMisiETQ6YTNwqunsKwJM4AkhfSVWCDkrCQd8EIgg4wJ7dmTrHoxYKvaFIRSrZMrgAQFhWKIULoCpHqESzixDQzTE/CDVLiHbSvPagQ5o3/1nGYSj+vfNU0ubEmU1hRjOzRLsiTwGXYP8Qfu1wy6D0Yv6t6nNHvXXNaqdMhqbxWkVrvURnhBw+moznMqVjhafqomIdhA34//gnpGa1s/65+K8PV7Gp0qPkdlnQo8iT3MgKmErYnNRtwzFyQxBEsO6JRUtZzIHS83UUHEpgntanfGFWroVC2FE8aK/UmdxHgCP6JYcuJARTuaSOC+2HX0F0luTGBK/vtQBY4X7TJPWMfoGzpw9K7ImsDR+0vNBAgeUZ6OCRErOBSRKQLOJmr99aBlsQqGvF2voikhgmgX4CDRc0wCbU6W0QnMVkudbIkcbHfDXG3hD1mSuKfTdsTlotMRHOwnNCSKWA8I4qJLs04FZFPPQQsnk5puhVdWWlPu+pAifNYkWaGHMu7qM6hIgMsXsFMsKgCm7AQHOQdJHVOJmdBGPREDpF1uZFEmXov3cio6WDcyeLyKYG0mfMRdljZ1EGJ7oUdQ8cVARpxCkUAPAAjkfsBKJ/v8u6szmLFdsoCTXYNmyqocroNDFMmcRoaMZbJsY5rJKSvL2J1NwulKkpUG0mrzrvQ5WvJbAk7g7uuva3e9xsZojgwUZQ4sBRoqhApHxCVAbU9IxxCqs+tO+hEhcZvbrWVVDCYRCWDZcKkYx03CGSICMFHivsCahJkRpykCs8VLaercZOTl9+aQp9k4Dl2gq0PnR+702F5ubRMz+VcQNGPwMYrqBwD6p0ptLaxwnJD61EyF8T1aDW2nLOij8XcYfnE3N5GM0FCltekrym1PsarHezx6rMvTmrxN9HrW6AaI0NPXTdzMD9lTOgwsTSmwHIiKNdQ+aWuIh2jCbWRqOsyWtNRI0MjfxpwypTfuoj5JlBhM5pXRlxIkXgJmJZAcFoiKbrr53e+XJLftkhMRSfZlp9Pvixtj8yRlcxfk9BvgnHg5mLhIcwU6CcuLLQ+9bTsuANdGVTF2+IDUi/1Pcg0YSYTnSZculQ0LPTMME2sQNhT9buJQ7JbnIUQSttKGVRb0hshzfRJXMRqglGeJfRCci2BIiJ/c3qcYBImqrCvu+hAQh4lG2ZIlNlYFOEjyWNoqeIC7G1dO6LvPmBiQ27AZ1IaKC4lFCwhLhxmFstrtLvvRF4absrO2JbOnNSXyfXhsIxuG5prhiVnJQ5DycmYMl8qvlFT72E3CXQ1NOVm7pGBsqfC0JuKDGLDeneGbpNEfoVdlzLiDz00iSUQ59bPd3lzljThrc4nRUoMLPPOWAifyMaRZGcmkWzmAHYpAeqUAqFmIVZPd3hvFGrdw9nsi1/23TOKyE4rXZ0JZRkD/VGcOnq1MhZHgI1EBxN2j12Xnuy4pQ2jR40la6qwVqKoi1DynzFOzK9TN1ZMoaPJDHSUs9hLzPScVN6fd2azhLnnJVStpoTTpXq4V5otPdgwX+izZ8Q1i9WAjiIQRQsDKzSpcnO0+w7UQXvd4pmNRaKZJnZNgpe+ggSLjoRka8RopbKMIKlMTN0K6kTmS276DJ8OCPE4siHEaUNp0J2nDvBGNJIpWkQ7SOk3K3yMMUFG6mApGavRUlt7vMProSk2koD0hbk6dhkqEIeCrxHJsAGNAmj4EVeMgGMdhHHM2q6QrFmalXHBso8ONMCcaRpJYcXLqR2UXEKQYODlUuLRcvwjKt4RuFpMcDPDf7kIxDLvoONiJQFbgNHl9lRhwZqdKuFsJj4/mAowCK0ixKXUBuQU7GvmWCjS16576UAEeOI6U+zXCpXDkualoCMPzTxHA1WOyIEoYdVIrTrBxIacawQGIT3TkqhQ62M3FVDYo02HGzGRnnNsmvA+OysFT6NeGhE8SaLo/LKaVEHwUOrUeI5r/XWghyPboCet9kO6CXmJ2qVAGE7JmPhEvIaNHCM87AGvhyadx+IyhoViPSCt65KhoZcPN3DZPq0rrKrANjgJpw+MJPFwYchZF2V4N2y1XKvVh/QY5FwMYezSuNnhbqooP7fUNjBlcYOW70NKvFovHlnWhLp9lmBZ+6RDE8Ngg4kduSzL8zJg2qmaAT2uSWY2iXG5hmiN09pMxmieTsys0ARclP4MJigRe5sGRE9j4kiXRlRs1UUHEghMbYRsdTxcUiHdFDive+uI1Xq5tROiSLAuqECkjk4WLBuGRfK45LBVHx2oYMnrzYHI4xrIwCM5jxgyZGpqBsHAjB/m6XH4Y8aSoTBFmcKY18/vfj+Rgu0cVMGCB5rPFG8ix1q6qxfP3PDYLClrk6Qu0wBXSCyGlPwJWTdF5XjZeQfioHpFbYYTl25M8BvXApIbZgwmTdqO6A4NmDScEeOSSASbZxkwb9ZlTtK5rGv13oE42H7ZlphpSV5GP1uLPPrepFATWIk4O7CTY8on/o5Z37BCMGigOKvqIJnOFGVlxfO1f0qcLvmmNwj7lGEyUGOJ1hNMORixJFwRUZ7D5osd03eNaC0mqMLvHejgkRo1kSmiKrMSvk6CjXgpA/9tZxUT7qx4z5yB5mA5ZzHDSATFsbYKxy31Hk3tugNdzjaTmsuIkfdSNVVMI5JAvL9MgE8gqcCUIXSswKqMxNMcnaswZqMKMRJ2FKZgqPmGz8Nm5ThTQqYtgAb9QCkHAKKKZYuN4KwcRzTl6RIR7R2JYqUSte5UdqdBSQ3dpJGK3P9NVxPMH54UgLGBRepYr4CQUKpcORXN3IjrLo1qPXR4feSNkXkDeEY9xWc+GpFw3MVikoDhB9QnQ8FZMYkrY8WECDpnHsXEa2quRL2TDmRgE4Qbi4OAnMgnGtgw8QgEr2UZJiwFBG0yRyWTGkSWxfsD2LBhxmQ4m/vlv+6oCykeBXjrpAIiihMsds1SISKaZfQipZ+mGwYL9huYpZN0fewMHv/BqH2WajEnycCnx14kGtHBdI/lHUOTxH5vlCuyKQG8olNY09zzgs5q6SCC4spY6CkEX8Rsx7FRrJbkRWOnGTpFqvLcVfd7UNeUulhFpRzwQGEyvpw4+tAxZhATTg9dFhoeeByxukmeVHtfH9/5dqxQqMBNf4hHYysVHjIGnSfLWfedlaSgJUSSQ5URcMQEj0iD2NH6+d1vprwpv7uumDLGiG3sIf1bqlZqQbCEtQqDnMXpWF5QzDnJ7YH+zHNEkjgqgUdlb7tpwjpNfFVGjdmWkRUv5dX9LMXFpNpKTH3MSr6BEQEfpzAznIVIgD1FZLDActy6j5004Mk81BqMrVRsj1G3jVx1uqp4IBMDkyykQPLAbMEEaHTycAEpx1jvoRMJ2G9tTUzQRHocRCvLQNRdPWbGMpWRMChMidNiRzAtIdgJawBFsZZmkkw26a0TNdY0AY1eQZcVWv2PtgpzccIISkQCy0cT9xkjYN1VOSLGVeU/OuYPxFmYxn4szr+YTS+qRaHRFp+batWVhzdBSmcCfIHqQakUM6JH1E8i+RGpIIJqHXSiIIvN5vdHPqNTwWgm8iEj5pQy+8QSg5cpZJ6OZZauYbAkzmzqkb9Rx4RO4mWiZFvF2cRjSunHEUnna/JhsAPweZ5qRJiJOPppJcH8T3NxZOaxpuLUeulEhzcca85CE/kCPk1fnVQcDSLX0qMYPsfmyQhSNAysOc0wZmkWyBtsJBY2Ez4eR2XnHSiDbqq1qf55q5K3DW5L3QYKXQQLO6WbgnopgWosuSDrOipXMWsLdxw6U8KCb0WgTm+m/jWpdEO9wlkeFGfp3WLUD7POIzHwQ8WBli6y3aofeBIbmSqqrjjv1JH94lGgzLLC+paSVSYWvpfmGfZ6ZBhkYvEfV7I9t9OJw+MYnjx7+eLZy9HrZ69eP/n+6bPfn7z84baRJqcf7qMaMDYDnhgEOy+lXiDoh5QQXUl0iXg6VKmaxWmXI4I2SH019LC3vULNqtt6cQ8bh3Yx61dAmxUwFsUdC4oyRAWVLcu7w+K2UfnN35+sd+5+uyPXGEmszlULLQgkYrMwOTyiROElBz5NlYSYQvHKONX9bzOi3377+tZbJVVkuI9hwXAMWJyUZ6Hk0JUsYz1WagRiIydpZCvpnOyK6Gyj9KdnDdm0J2xHlqnHVdPVmGQsqpiyKpV40KBrMXkzd9C3jZRNsmrLdBnU1988+f3rr+9gPCguXwYnYPGdJLEp8WmCek8CcMaYB4VhUxmz16FPQtnTu5Srq5OWMLEzY6moJMxLkBzB+4nkDuQ+8XUf2r59dgfajtYhZpnkHCa6JWLEhankcjAfB2PJ8oiG6nvlcTKmyxIkhc9+eH4HCn1YWwUE7CRHdwYLyIBzG6cox1iPfIRlJUqMrT3Rgb7nL14/e/nkLoMoqkmZogklN2LYmYn5sDmKvxgNtuVQeEIW74jD0OpVHnBIfs6MNJsmznUl+YcnL5/dGpVzFPl8Yl/QCwPlGL2naS0nFxJcZmkXEKnhHLFoFXIo3hl1KWn86cVP/3GXQfVndEVaOPkoZt0mHo4WGMe6B47ltDGWPHeA59CxDIdEPCTjJu66OEHld3eiEmSqCOcUevkYYp4JDsnobIUdXPwl0lA8w73OSag2ifSqanOhoKuhb3c694+E/+Pvpp5ovh/R1voIZOTHlsOWaMYSFEcJgbG4nyFAFLueThWFPISdEtFLIn/++ee/32F0dZ16NxZrfiWQ0NgxMJglZs7jZ4lBholteWpjqQV2p+4uc6+hzJK6JIdoYcZ6Kqm/Ql0K8yplTSfonZottyd5r+6FvJB1LhPLDcS6x9YHnTjVUoCNmR2VWnl9gPvRPR685k9q6U9e2+vktcnVTGksT1z7IE9na5/H05/P1p/P1p/P1rfr2pbz395MVzzdcXp2ceeT37TRo3f9+W9M7nDN898M89r6898eot31/Lf1YnnPJ7+xCJU/X+TuCm8k59wTLRvBzrExIXkSsWJGGX2pTCg7rJ8KvDtRAOOhjufZFHrDrFhBQA8gaSdXi3f7Bmt9moYifeRcozwkhgImg2AoYcikBGLgCktlSEFMPpF0zBxiifr4DkOpKGnji70zwB1FLC0f5FFoMsmFwbUkzWzqApjskfrbdAa65tFWRN59QDUHrCy2yTPF4thmNuCpTpZogYQ1c+h2CzKpERuX50ZFpnOlAu/Wv60jSxOO1C43BIzyzGzHI2YUN8uyirB9mTLG1HHG9HyNmbhr7YuSxLsPaBwptVrCLQ9iZ5gqxmw2njhFD1ZCZI5lYjfR+/RjlzW+OhZeyFMPg7vliPoqhaH11fZtRkheluZZLpB54wxt2ySLbZLANpfzkXw0p2PVgYpEqO+T4mK1L4U+g1aziegmyORsDPoDfZljXENjRUgF0Dkf7ewKq89gk5LAk+JsUWC6J8X5cnq1XAPnxI2cCLLfOvkR8XCl3FmeeM7qyAy/seBuwtAJ6MhzOcZL3RxdMYU8bUFwVscQtEs6ya9Wxb7eSs1Wj3xKZcAEHWb9CpTdGD2EiZd58gQBsTlzgfSyhle61hFgfTkl9/pxM0qQjpv8wKKxhnFqw0NFTCxoVR7iwPOGEpbOkaMLah10wIWQnd5F6HnPop6WRsSay/M0Y9EmOc6FUH2IFR52AHaTpJKGqsuyK4CkIvEuUxsGHjEhP4Lcsg4+jy/JrEB7mXqVZcwtdCy7I6ehevaXd9usllL0LvqDr1fpPaaBI+Y2jfOMdQITibzyzAieT8vSq1pcS7GI3fIa1gQeLzCSv9xyKDWGH5Xn4RqTcs+y3lIk9W8sq9KzsmEWZjJ5/vCnnUcD0mNDL8+kWFwUi8cnk/loejG/Wo0WxeloevI7U8NilyR3Cuf4gITPDWRKYEgOTb4cCqrK8DiriMUEsKaJKInLg0RNp5ht7QuKN2DZo8nJ76yvi7Xv7kc2+kXs0/CJhjR0WfMQk1xqwSY8GovHNPFYQqcHGcmcpdev5Pv0D6pO37sH93IPvn/X39oU651+vdOvd/r1rUOj/28GnrWcjO/L3bfRbvb/hcwnKv1/MLTMZ6GxsQt7/99DtP39f1wtN7n62jrI0+evXj9/gT8UJxCRq3fzQlSGAkKsOJ2+taVU23hjIHfJU+fT5Qr3KYvxagVY1cGQxJy+Iw2jN7EdQbXcwrgH69ti3GZGBirpagruN1oWb8lVT8+nk9Xw0efQoiYt8s+L09XBojgr3s7x1Px8DPF/NYdMrmsQFPP//J+vj34Oj/L/HnrBb4XVzm2NpNPzy8vFwezyAtr76LhY/VYUFweTSaCa5sl4BbJmGJ95cDxdrH6RK48GX0BZlL7GKjB9mwfKxEcn0yXoesc79MoORQEfjEGdXl4MJhPcKly/eetyPoXGoorHYE6BMA8qNQQ8HyTXtRKdF1zcpoXpRD1/8vRIweH4U/UHXDr6+ruhClJKlUYfxYbM3hiqhqyptJ61JC6CatUpHZzwGbS40eXF+TtBY+LCabG+1JT8Iol5z1Clcb0/yGUO+XpZl0+qbvXvXtLOxqvJL8VJ9dtS3Wku1A1qoRNjukcXxW/Hl4uLoX5+gEvUFoxXEbr1MceVMbTUSaMXmPZ79YLvgFk0Xb3TXg60m68GOZlH2WcaP9qr0/HJ1fmqTtZXA5N07qE5mb4X3QL89TcyrcO9uqsvhY0Ov7u2u22qmjAV1a8muskK7qQiqHM/bqZgbr32pGNQX/UY2zqV//X8P58Jof82XKtW5svPb83/l9iosGlG1APuUaw0mtS8cMl18p9tLf+T9DMeARvZzwbJH0VQvX3i8r8+//hvNFlcLpfYBZCNF8UYsuVqvg72VX84nV6Qw3dUGHfofyx50Zz/iKfM9vrfQ7TnL149e/kasvH191s0sJN/jSC7Jr+WYCIvX4db9D+Rpf86f7xlAT2+dgENvYXqpTVF4y/TFQxi8FIVvafj6bmXvUPiFIbkserAKB05/iE6iw4r2Xs4OB2fL6FKkYv6/+Op2Xh+MFxcnRdC7hgKxIEqMocDccIciXEKCsHzsSxOxu8eDeggg9CNBvyo5WC8kr9A6VpNLyarAdUQrKHZfPnloHg7nqyOVsvB2eLyao57F8Xg6Y92UBsUjEelpA7x3JmQogMqW3Eozsjh0x8Zl5lcLRbUd6qXHDxSiVLTUNfjpx9fKdxek+JrN30MlcbVfKapXuljm/4IyjiM+S49szXblaa4ViXbCt4N2t1gUAm86vkO/rzDwepSOjxofRrUxfe9+T6Athf/n87GZ1TsscxnezgLdvL/tM3/mTvX8/+HaA/L/+sLqOf9HzPvr890z/c/3LYX/78lMnQX/4/TsM3/rYt6/v8Q7WH5/3oB9dz/Y+b+tVD0H8L7BQOwVQTcEPHfGmr3+In6c5+U6Gjzfx9SKN2Xa96PeZ4UJ1eLYv8o4a74X2xNi/9bix89/3+Adu/8v7WAHm8uoBbrr6Ilf61Y//LX6XxYsqxKPgiyYX8JoY/tLyOe/vh0MD8fXy2nx+fFtxoDOHhjBufTU8ipN8tBI0jWDMV9OWCo4AsNQHjWtRxoyKmSAMK1VAScjKIgC/ISdnR0DsZzPnjy6h8EaLoBSVr+YbJjuRFI/Opvg7Og+nbPE7bLAhndUhocCIX+BWuGXRc+NwLPqiXig6obsdz7BaNtdH8ncBpHgzdpSHRHlmb5objz0aAKwh74oZNu1u86xMJ8e9CeD40uty5Wn3RtSF07XT8nMKzqMZXVcm2ER5YMEmMA3hgNhm2siaaw1laJ7NZnPBqcyT3fv1BQ0lnQuqEmjyUsth7NXSHq5bzsedkKVNf+xg/YXOsSFZbA3mEVMfwYBf01rS3/j7Gf/68YLceM9I7mBDieXi5mI7DXYrmckatMsSvu0/7jaQlt+y9Je/n/IO3e5X9jAT2+dgG1lIBZMJ7AoiI7oVlQk+8NNeA4uKBw8ndfd9cyuBgtLn9bQlXgAzuUhTOC6SEmD8bHywPef7TWH/RF5PRg99WN4eH6FUd8heLHy2s3KRd+dHhbiRrnK2uGSCnkxyez6WoFs3F+frUcFW8n51cnxUn5kB+BLc9dXK68+hIPFsXRrBgvIWcGV8KusdbBvAfHi/Hk12L1pVioOrWD6QVMoylNnZPB8buBEjqIwuzob9aYAbjtYHb5pphJFtM9aCIVQr2OlZEhVdGl3P5/N0QnF9PndWFVs3UpqqoVN3w0WH7+5OX3r141hWvjja2X6VcHmKrrWd+jwfG2bqlqgvyRdK9oLcxQQ8XY+mGL4rTAEE2IhZlNl5S4I6YXTCfLxsP6seJImDY/dGNrtfQayr3F5dWq8A/KM5j1OdHEo+ICr/pFpvXRYPapiL2++daW/wRl1UGRo03U4KiBKeugCezy/4Ymasn/1CZJL/8fot27/G8voMebC+hxE5TY1AQqwOSN/oA1wK6LP2B992ENkHmDmK4Ad/yqFrWDoXTAP0wnJyaczEos32gyvuL3bbPxN4HKd5Ge+6OzVXzcAA3tOf8n2Tbif5cXq8X4BFuFnH5bmY9j6Ebj6eL+8B8RFmnb/nO2z/94kHb/8b/6AtoW+XvsF9CWCODx+GR//y4f2t+7SwogY5qBwHYaRiOMRnfgJWTI4Ku/DsA09S81Vv9uKW5A2nVD1sANzWGtRgmeaP7apIdVxY0/yrF7PBdi8WOr7xYjt8Nze9OIlFTX3lfP3KBbTYdh/cXDzcTQoeZaj+RF5WR7vyDt3ZJWztfodDybnlcJuXSKdnp5OQ0bSZ1CwY0ZuCWgH0O5x2tN6l8r03uH12LqdvqSG7HWllF802isZ2WD1m1B15tWQn12xKPticDCawd7T5rXPh1H6wfabmv/SeLO/eD/Q+fSDfx/3ON/HqS9N/tPM7/+DLafp/Sjsvv0mz6//f73eYmdOMBO/08Yt/0/YY//fpj23vZ/mdj6Z+AAFa0fFQ8ov6rXwD7hdrv8z+Xkcg8g4E78t2vH/5Mo7P3/D9LeU/6nLKAeA/4xY8Cbc93jwD/Mdlv9v6oG8sfEf+Oo5/8P0t6b/r8uJ/NnsABq1H5UNsD6uz4ajta3fVrn/J9q9z5Q/k/v/3mQ9nD5P9UC6vN/WrKhz//x+T9rFfMjz/+pPrTP/+nzf95r28//dzsdYGf9n8S18V9R1Of/PEh7YP/fNTpA7/r7uFx/NfHWV/75oNtt/H/4/b3W/8G/ff2f99Tei/+P9bQ/dK+f0HhHX9+N1ebfvwOQn9hzxE+77aX/37IG0G30/z7/42Haw+r/19UA6vX/j0r/r5e36fX/vvWtb33rW9/61re+9a1vfetb3/rWt771rW9961vfHrz9P3jrYQYAGAEA"""

def qs(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

def adapt(value):
    return (value
        .replace("dq4_silver_20260825", RUN)
        .replace("2026-08-25T10:50:49.897Z", RUN_OPEN_TS)
        .replace("f5c7c7ab-e37d-4a31-b9c2-b7631becb16a", SILVER_UPDATE_ID))

def unpack():
    archive = tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz")
    items = []
    for member in archive.getmembers():
        if member.isfile() and member.name.endswith(".sql"):
            items.append((member.name, adapt(archive.extractfile(member).read().decode("utf-8"))))
    return sorted(items, key=lambda x: (0 if "/pass2_" in x[0] else 1 if x[0].endswith("/mce_scan.sql") else 2, x[0]))

def execute(seq, name, sql):
    sha = hashlib.sha256(sql.encode()).hexdigest()
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(sql).collect()
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'ok',NULL,NULL,current_timestamp(),current_timestamp(),'DQ4')""")
    except Exception as exc:
        msg = str(exc)[:4000]
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'error',{qs(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
        raise

run_row = spark.sql(f"SELECT count(*) n FROM 8_dev.silver_qc.dq_run WHERE run_id={qs(RUN)} AND finished_at IS NULL").first().n
assert run_row == 1, "run_id must identify one open dq_run row"

items = unpack()
for seq, (name, sql) in enumerate(items):
    execute(seq, name, sql)

print({"run_id": RUN, "lane": LANE, "statements": len(items), "status": "ok"})